In [0]:
%run ../../config/utils

In [0]:
import sys
sys.path.append("..")
sys.path.append("../..")

from lib.job_manager import load_config, split_config
from lib_etl.s3 import etl_input_data_validator
import lib_etl.validations_ETL as validations
from lib.misc import get_latest_path
from pyspark.sql.types import *

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)

## Load source data

In [0]:
EXCLUSION_SCHEMA = StructType(
    [
        StructField("CATEGORY_TYPE", StringType(), False),
        StructField("CATEGORY_CD", LongType(), False),
        StructField("CATEGORY_DESCRIPTION", StringType(), False),
        StructField("INCLUDE_OR_EXCLUDE", StringType(), False),
        StructField("EXCLUSION_TYPE", StringType(), False),
        StructField("EXCLUSION_SUBTYPE", StringType(), False),
        StructField("SEASON_MONTH_1", LongType(), False),
        StructField("SEASON_MONTH_2", LongType(), False),
        StructField("SEASON_MONTH_3", LongType(), False),
        StructField("SEASON_MONTH_4", LongType(), False),
        StructField("SEASON_MONTH_5", LongType(), False),
        StructField("SEASON_MONTH_6", LongType(), False),
        StructField("SEASON_MONTH_7", LongType(), False),
        StructField("SEASON_MONTH_8", LongType(), False),
        StructField("SEASON_MONTH_9", LongType(), False),
        StructField("SEASON_MONTH_10", LongType(), False),
        StructField("SEASON_MONTH_11", LongType(), False),
        StructField("SEASON_MONTH_12", LongType(), False),
    ]
)

In [0]:
df = spark.read.csv(exclusions_path, header=True, schema=EXCLUSION_SCHEMA)
df.createOrReplaceTempView('df')

## Save to delta table

In [0]:
spark.sql(f"""
    INSERT OVERWRITE {bronze_exclusions_brand_exclusions_mixed}
    SELECT
        CATEGORY_TYPE,
        CATEGORY_CD,
        CATEGORY_DESCRIPTION,
        INCLUDE_OR_EXCLUDE,
        EXCLUSION_TYPE,
        EXCLUSION_SUBTYPE,
        SEASON_MONTH_1,
        SEASON_MONTH_2,
        SEASON_MONTH_3,
        SEASON_MONTH_4,
        SEASON_MONTH_5,
        SEASON_MONTH_6,
        SEASON_MONTH_7,
        SEASON_MONTH_8,
        SEASON_MONTH_9,
        SEASON_MONTH_10,
        SEASON_MONTH_11,
        SEASON_MONTH_12
    FROM df
""")